# Passo a passo do workshop 

Este é um workshop para aprender como o  Databricks Apps pode subir e mostrar imagens de volumes, ler do datalake usando o SQL Warehouse e escrevendo/lendo do Lakebase.

Este é um guia para subir o **Databricks App** com Unity Catalog + SQL Warehouse, e (opcionalmente) integrar **Lakebase Autoscaling**.

O Databricks app em Streamlit tem três abas:

| Aba | Backend |
|-----|--------|
| Photo Album | Unity Catalog Volume |
| Cliente Table | SQL Warehouse + tabela Delta |
| Dados Lakebase | Lakebase Autoscaling (Postgres) |

---

Este notebooks possui o seguinte conteúdo


| Seção | Conteúdo |
|-------|----------|
| **Seção 1** | UC + SQL Warehouse + App + `app.yaml` base + deploy + share |
| **Seção 2** | Lakebase Autoscaling + recurso `postgres` + role/GRANTs |


## Execute as células abaixo para configurar os valores do seu ambiente

In [0]:
dbutils.widgets.removeAll()

In [0]:
# Widgets de configuração do notebook
# Estes valores serão usados nas células SQL abaixo
     

dbutils.widgets.removeAll()

dbutils.widgets.text("nome_catalogo","","Catálogo")
dbutils.widgets.text("app_id","","App Id")

# Captura o nome do catálogo do widget
nome_catalogo = dbutils.widgets.get("nome_catalogo")
app_id = dbutils.widgets.get("app_id")

print(f"📋 Configuração:")
print(f"Catálogo: {nome_catalogo}")
print(f"App Id: {app_id}")

📋 Configuração:
Catálogo: workshop_app2
App Id: fac9da28-6611-459f-95d5-99f3be52b88d


---
# Seção 1 — SQL Warehouse + Databricks App

Passos para ter o app rodando com **Photo Album** e **Cliente Table**.

---
## 1.1 Unity Catalog — catalog, schema, tabela e volume

Vamos configurar e subir dados fictícios no catálogo e schema da sua escolha. Estes dados serão usados
pelas abas do app **Photo Album** e **Cliente Table**.

> ⚠️ **AVISO** 
> **Substitua** os valores do nome do catálogo e execute a célula SQL abaixo (ajuste nomes se necessário).




In [0]:
%sql
-- Usa o widget nome_catalogo para criar os objetos do Unity Catalog

-- Descomente a linha abaixo se precisar criar o catálogo:
-- CREATE CATALOG IF NOT EXISTS `${nome_catalogo}`

-- Cria no catalogo um schema chamado default
CREATE SCHEMA IF NOT EXISTS `${nome_catalogo}`.default
  COMMENT 'Schema workshop';

-- Cria no schema uma tabela chamada clientes
CREATE TABLE IF NOT EXISTS `${nome_catalogo}`.default.clientes (
  id INT,
  nome STRING,
  cidade STRING,
  segmento STRING
) COMMENT 'Clientes demo';

-- Insere dados de exemplo
INSERT INTO `${nome_catalogo}`.default.clientes
VALUES
  (1, 'Empresa Alpha', 'São Paulo', 'Industrial'),
  (2, 'Beta Energia', 'Rio de Janeiro', 'Comercial'),
  (3, 'Gamma Solar', 'Curitiba', 'Renovável');

-- Cria um volume para armazenar as fotos
CREATE VOLUME IF NOT EXISTS `${nome_catalogo}`.default.fotos
  COMMENT 'Fotos do album demo';


In [0]:
print("⚠️ Paths resultantes (use no `app.yaml`):")

# Substitua abaixo pelo valor do widget nome_catalogo
nome_catalogo = dbutils.widgets.get("nome_catalogo")

VOLUME_PATH = f"/Volumes/{nome_catalogo}/default/fotos"
CLIENTES_TABLE = f"{nome_catalogo}.default.clientes"

print("VOLUME_PATH =", VOLUME_PATH)
print("CLIENTES_TABLE =", CLIENTES_TABLE)

⚠️ Paths resultantes (use no `app.yaml`):
VOLUME_PATH = /Volumes/workshop_app2/default/fotos
CLIENTES_TABLE = workshop_app2.default.clientes


---
## 1.2 Obter o ID do SQL Warehouse

1. **Compute** → **SQL Warehouses** → selecione (ou crie Serverless).
2. **Overview** → copie o **ID** → esse valor vai em `DATABRICKS_WAREHOUSE_ID` no `app.yaml`.
3. Clique **Start** se estiver parado.


---
## 1.3 Criar o Databricks App

1. **Compute/Databricks Apps → Apps → Create custom app**
2. Nome: `app-album` (só minúsculas, números e hífens)
3. Create
4. Esperar entre 2-3 min até que o app suba e fique no status **Unavailable**.

### Anotar o App Id
 1. Vá para a  página de detalhes do app;
 2. Procure do lado esquerdo **About the app**
 3. ⚠️ Copie o **App Id**. Algo como xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx


---
## 1.4 Configurar `app.yaml` (base — sem Lakebase)

Para a Seção 1, basta warehouse + volume + tabela. **Não é obrigatório** ter `LAKEBASE_ENDPOINT` ainda.

### Template mínimo (Seção 1)

```yaml
command:
  - streamlit
  - run
  - app.py
env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: "xxxxxxx"                                    
  - name: VOLUME_PATH
    value: "/Volumes/xxxxxxx_catalog/default/fotos"     
  - name: CLIENTES_TABLE
    value: "xxxxxxx_catalog.default.clientes"           
permissions:
  - permission: CAN_USE
    level: APP
    resource_type: servingEndpoints
  - permission: CAN_USE
    level: APP
    resource_type: sql
```

## Passo a passo no app.yaml
1. Navegue até a folder do repositório clonado do git.
2. Vá para a para **/app**.
3. Abra o arquivo **app.yaml**.
4. ⚠️ Substitua os valores coletados nas seções anteriores. Caso você veja a opção **LAKEBASE_ENDPOINT** deixe-a como está.


---
## 1.5 Permissões ( SQL Warehouse + Unity Catalog)

O App precisa de permissão para acessar o SQL Warehouse e o Unity Catalog. 



### SQL Warehouse → CAN_USE
1. Navegue até **Compute** → **SQL Warehouses** → selecione o SQL Warehouse usado no app.
2. Clique na aba de **Permissions**
3. Cole aqui o service principal (**App Id** copiado nas seções anteriores)
4. Na lista de app, selecione o **app-xxxxxxx** que corresponde ao **app-album**.
5. Selecione a permissão de **Can use**.
6. Clique em **Add**.


### Unity Catalog → GRANTs
1. ⚠️  Substitua o **App Id** no widget (no espaço em branco do notebook).
2. Execute a célula SQL abaixo.


In [0]:
%sql
-- Dá para permissão para o APP de ler da tabela de cliente e ler/escrever no volume fotos

GRANT USE CATALOG ON CATALOG `${nome_catalogo}`
  TO `${app_id}`;

GRANT USE SCHEMA ON SCHEMA`${nome_catalogo}`.default
  TO `${app_id}`;

GRANT SELECT ON TABLE `${nome_catalogo}`.default.clientes
  TO `${app_id}`;

GRANT READ VOLUME, WRITE VOLUME
  ON VOLUME `${nome_catalogo}`.default.fotos
  TO `${app_id}`;


---
## 1.6 Deploy do código e do App
1. **Compute/Databricks apps → Apps →** `$APP_NAME` → **Deploy**
2. Selecione o caminho da pasta **/app** copiada/clona do git 
3. Aguarde **Running** (1–3 min)
4. Abra a URL
5. Verifique o funcionamento das abas **Cliente Table** e **Photo Album**.
6. Tente fazer upload de imagens, faça o reload da página e observe como as imagens ficam persistidas como se fosse um albúm de fotos.


---
# Seção 2 — Integração Lakebase Autoscaling (opcional)
 
 Esta seção apresenta como conectar o Databricks Apps o **Lakebase**. Na aba **Dados Lakebase** do app é possíve inserir e alterar dados do Lakebase (CRUD em Postgres).

⚠️ Pré-requisito: Seção 1 concluída (app criado, SP conhecido, código sincronizado).

### Como a autenticação funciona

1. Anexa-se o Lakebase como recurso `postgres` → runtime injeta `PGHOST`, `PGUSER`, `PGDATABASE`, `PGPORT`, `PGSSLMODE`.
2. No `app.yaml`, `LAKEBASE_ENDPOINT` usa `valueFrom: postgres`.
3. No código: `w.postgres.generate_database_credential(endpoint=...)` gera token OAuth (~60 min) como senha.
4. Username Postgres = Client ID do SP do app.

> Use a chave **`postgres`** (Autoscaling). A chave legada `database` é do tier Provisioned.
>
> Requer `databricks-sdk>=0.89.0` no `requirements.txt` (senão: `WorkspaceClient has no attribute 'postgres'`).


---
## 2.1 Criar o projeto Lakebase (do zero)
1. Navegue até **Lakebase Postgres**
2. Clique em **New Project**
3. Dê o nome de "lakebase-album"
4. Postgres 17
5. **Create** 
6. Seu Lakebase já foi criado.

---
## 2.2 Criar a tabela no SQL Editor do Lakebase

1. Abra o **Lakebase Project** criado na etapa anterior.
2. Navegue até a aba **SQL Editor**.
3. Selecione o banco de dados `databricks_postgres`.
4. Cole e execute o SQL abaixo:



⚠️ CREATE TABLE IF NOT EXISTS public.registros (
  id     SERIAL PRIMARY KEY,
  status TEXT NOT NULL
);


---
## 2.3 Atualizar/Checar `app.yaml` para Lakebase

Acrescente (ou confirme) no `env` do **app.yaml**:

```yaml
  - name: LAKEBASE_ENDPOINT
    valueFrom: postgres    
```

### Variáveis Lakebase — origem

| Variável | Onde | Origem |
|----------|------|--------|
| `LAKEBASE_ENDPOINT` | `app.yaml` (`valueFrom: postgres`) | Path do endpoint, resolvido pelo runtime após anexar o recurso |
| `PGHOST` | **Não** no yaml | Injetada pelo recurso `postgres` |
| `PGDATABASE` | **Não** no yaml | Injetada (`databricks_postgres`) |
| `PGPORT` | **Não** no yaml | Injetada (`5432`) |
| `PGUSER` | **Não** no yaml | Injetada = Client ID do SP do app |
| `PGSSLMODE` | **Não** no yaml | Injetada (`require`) |

**Não** hardcodar host/secret no yaml.  
**Não** usar `resource_type: lakebase` em `permissions` — o bind é via App resources.


---
## 2.4 Anexar o recurso `postgres` ao app
1. **Compute/Databricks Apps → Apps →** `$APP_NAME` → **Settings** (aba do lado esquerdo)
2. **+ Add resource → Database** → Lakebase Autoscaling
3. Selecione project / branch / database (ex: databricks_postgres)
4. Permissão: **Can connect and create**
5. Resource key: **`postgres`** (igual ao `valueFrom`)
6. **Save**



---
## 2.5  GRANTs na tabela `registros` e permissão para o Databricks Apps acessar o Lakebase

1. Abra o **Lakebase Project** criado na etapa anterior.
2. Navegue até a aba **SQL Editor**.
3. Selecione o banco de dados `databricks_postgres`.
4. Execute a célula baixo e copie o output.
5. ⚠️ Com o output copiado, no SQL Editor do Lakebase, execute os comandos gerados:

In [0]:
# Gera o SQL para executar no Lakebase SQL Editor
# Copie o output e cole no SQL Editor do Lakebase

app_id = dbutils.widgets.get("app_id")

sql_lakebase = f"""
-- Cria a role se não existir
DO $$
BEGIN
   IF NOT EXISTS (
      SELECT 1 FROM pg_roles WHERE rolname = '{app_id}'
   ) THEN
      CREATE ROLE \"{app_id}\";
   END IF;
END
$$;

-- GRANTs para a tabela registros
GRANT USAGE ON SCHEMA public TO \"{app_id}\";
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLE public.registros TO \"{app_id}\";
GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO \"{app_id}\";
"""

print("⚠️ Copie o SQL abaixo e cole no SQL Editor do Lakebase:")
print("="*60)
print(sql_lakebase)
print("="*60)

⚠️ Copie o SQL abaixo e cole no SQL Editor do Lakebase:

-- Cria a role se não existir
DO $$
BEGIN
   IF NOT EXISTS (
      SELECT 1 FROM pg_roles WHERE rolname = 'fac9da28-6611-459f-95d5-99f3be52b88d'
   ) THEN
      CREATE ROLE "fac9da28-6611-459f-95d5-99f3be52b88d";
   END IF;
END
$$;

-- GRANTs para a tabela registros
GRANT USAGE ON SCHEMA public TO "fac9da28-6611-459f-95d5-99f3be52b88d";
GRANT SELECT, INSERT, UPDATE, DELETE ON TABLE public.registros TO "fac9da28-6611-459f-95d5-99f3be52b88d";
GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO "fac9da28-6611-459f-95d5-99f3be52b88d";



---
## 2.6 Sync + redeploy do Databricks Apps

1. Abra o Databricks Apps na interface web.
2. Navegue até o app desejado (**$APP_NAME**).
3. Clique no botão **Deploy**.
4. Aguarde o status **Running**.
5. Pronto! O app foi redeployado.

---
## 2.7 Validar aba Lakebase + troubleshooting

1. Abra a URL do app → aba **Dados Lakebase**
2. Expander **Debug** — confirme `LAKEBASE_ENDPOINT` e `PGHOST` preenchidos
3. **Atualizar Lista** / adicionar um registro

| Sintoma | Causa | Correção |
|---------|-------|----------|
| `'WorkspaceClient' has no attribute 'postgres'` | SDK `< 0.89` | `databricks-sdk>=0.89.0` + redeploy |
| `PGHOST` / `LAKEBASE_ENDPOINT` vazios | Recurso não anexado ou `valueFrom` errado | S2.3 + S2.4 |
| Erro auth Postgres | Role do SP ausente | S2.5 `list-roles` / `create-role` |
| `relation "registros" does not exist` | Tabela não criada | S2.2 |
| `permission denied` | Falta GRANT | S2.5 GRANTs |

---

✅ **Fim da Seção 2 (opcional).**


---
# Seção 3 — Dashboard AI/BI Embedado (opcional)

Esta seção apresenta como embedar um **Dashboard AI/BI** dentro do Databricks App. Na aba **AI/BI Dashboard** do app será possível visualizar o dashboard diretamente embedado.

⚠️ Pré-requisito: Seção 1 concluída (app criado e rodando).

### Como o embed funciona

1. Criar e publicar um dashboard AI/BI
2. Obter o Dashboard ID da URL
3. Compartilhar o dashboard com o Service Principal do app
4. Configurar o Dashboard ID no `app.yaml`
5. Workspace admin precisa permitir embedding no Settings
6. URL de embed: `https://<workspace>/embed/dashboardsv3/<dashboard-id>`

---
## 3.1 Criar e publicar o Dashboard AI/BI

1. Navegue até **AI/BI → Dashboards**
2. Clique em **Create dashboard**
3. Dê um nome ao dashboard (ex: "Workshop Album Dashboard")
4. Adicione visualizações e widgets conforme necessário usando a tabela de **Clientes** usada anteriormente.
5. **IMPORTANTE**: Clique em **Publish** no canto superior direito
   - Apenas dashboards **publicados** podem ser embedados
   - Escolha as permissões de dados:
     - **Shared data permissions**: Queries rodam com as credenciais do publisher
     - **Individual data permissions**: Cada viewer precisa acesso aos dados

⚠️ **Nota**: O dashboard precisa estar **publicado** (não apenas salvo como draft) para funcionar o embed.

---
## 3.2 Obter o Dashboard ID da URL

1. Com o dashboard aberto no navegador, observe a URL:
   ```
   https://xxxxxxxxxxxx.cloud.databricks.com/sql/dashboards/yyyyyyyyyyyyyyyyyy
   ```

2. O **Dashboard ID** é a última parte da URL (após `/dashboards/`):
   ```
   yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
   ```

3. ⚠️ Copie este ID — você vai precisar dele no `app.yaml`

⚠️ **Dica**: O Dashboard ID é um UUID de 32 caracteres hexadecimais

---
## 3.3 Compartilhar o dashboard com o App (Service Principal)

1. No dashboard publicado, clique no botão **Share** (canto superior direito)
2. Na janela de compartilhamento:
   - ⚠️ Cole o **App ID** (Service Principal do Databricks Apps) na caixa de texto
   - O App ID é o mesmo usado nas seções anteriores (ex: `fac9da28-6611-459f-95d5-99f3be52b88d`)
3. Selecione a permissão **Can view**
4. Clique em **Add** ou **Share**

---
## 3.4 Configurar o Dashboard ID no `app.yaml`

Adicione a variável `DASHBOARD_ID` no arquivo `app.yaml`:

```yaml
env:
  # ... suas outras variáveis ...
  
  # Dashboard ID para embed na aba AI/BI Dashboard
  - name: DASHBOARD_ID
    value: "xxxxxxxxxxxxxxxxxxxxxxxxxx"  # ⚠️ Cole seu Dashboard ID aqui
```

---
## 3.5 Configurar permissões de embedding (Workspace Admin)

**Esta etapa requer permissões de Workspace Admin.**

Para permitir que dashboards sejam embedados em Databricks Apps:

### Passos:

1. Clique no seu **username** no canto superior direito
2. Selecione **Settings**
3. No menu lateral, clique em **Security**
4. Role até a seção **External access**
5. Encontre **Embed dashboards**
6. Configure a política de embedding:

#### Opção A: Permitir todos os domínios (mais simples)
   - Selecione **Allow** no dropdown
   - ✅ Dashboards podem ser embedados em qualquer domínio

#### Opção B: Permitir apenas domínios aprovados (recomendado para produção)
   - Selecione **Allow approved domains**
   - Clique em **Manage** ao lado de "Embed Dashboards"
   - Adicione os domínios abaixo:
   - Clique em **Add domain**
   - Clique em **Save**

### Domínios necessários para Databricks Apps:

```
*.databricksapps.com
*.aws.databricksapps.com
*.aws.databricksapps.com
```

⚠️ **Importante**: 
- Sem essa configuração, o dashboard **NÃO** carregará no iframe
- Você verá o erro: "refused to connect" ou uma página em branco
- Apenas admins podem fazer essa configuração

---
## 3.6 Deploy e validação

### Deploy do app:

1. Abra o Databricks Apps na interface web
2. Navegue até o app (`app-album`)
3. Clique no botão **Deploy**
4. Aguarde o status **Running**

### Validação:

1. Abra a URL do app
2. Navegue até a aba **📈 AI/BI Dashboard**
3. O dashboard deve aparecer embedado no iframe

### Troubleshooting:

| Sintoma | Causa provável | Correção |
|---------|----------------|----------|
| Dashboard não carrega / "refused to connect" | Admin não liberou embedding | Seção 3.5 — configurar Settings > Security |
| Página em branco | Dashboard ID incorreto | Verificar Dashboard ID no app.yaml (Seção 3.2) |
| "Permission denied" | Dashboard não compartilhado com o app | Seção 3.3 — Share com o App ID |
| Dashboard não publicado | Dashboard está em modo draft | Seção 3.1 — clicar em Publish |
| URL errada no código | Código usando `/sql/dashboards/` | Verificar que o código usa `/embed/dashboardsv3/` |

### Botão "Abrir Dashboard":

Se o embed não funcionar por algum motivo, o app tem um botão **"🔗 Abrir Dashboard"** que abre o dashboard em uma nova aba — isso sempre funciona!

---

✅ **Fim da Seção 3 (opcional).**

---
# Referências

### Seção 1 — Apps / SQL / UC
- [Databricks Apps](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/)
- [Apps — SQL Warehouse](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/sql-warehouse)
- [Apps — Permissions](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/permissions)

### Seção 2 — Lakebase (opcional)
- [Add a Lakebase resource to a Databricks app](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/lakebase)
- [Postgres API — generate database credential](https://docs.databricks.com/api/workspace/postgres/generatedatabasecredential)
- [Connect external app to Lakebase (SDK)](https://docs.databricks.com/aws/en/oltp/projects/external-apps-connect)
- SDK: `w.postgres` requer **databricks-sdk >= 0.89.0**

### Seção 3 — Dashboard Embedding (opcional)
- [Embed Databricks apps in web applications](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/embed/)
- [Embed a dashboard](https://docs.databricks.com/aws/en/dashboards/share/embedding/index/)
- [Basic dashboard embedding](https://docs.databricks.com/aws/en/dashboards/share/embedding/basic/)
- [Manage dashboard and Genie access](https://docs.databricks.com/aws/en/ai-bi/admin/embed/)
